# Дневник питания с умными проверками

Замкнутый цикл: **норма (под человека + болезнь) → выбор продуктов → подсчёт факта → предупреждения по болезни → подбор, чтобы добрать норму**.

Как пользоваться: заполни профиль в ячейке 1, потом повторяй цикл «поиск → добавить (ячейка 3) → посмотреть состояние (ячейка 4)».

> ⚠️ Расчёт ориентировочный, не заменяет врача/диетолога.

## 0. Импорты и загрузка базы

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from diet import (UserProfile, calculate, load_foods, search, to_fooditem,
                  FoodLog, check_portion, check_day, suggest, CONDITIONS)

pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 45)

foods = load_foods()
log = FoodLog()
print(f'База загружена: {len(foods)} продуктов')

## 1. Профиль человека и норма дня

Задай свои данные и болезнь. Норма считается существующим калькулятором.

In [ ]:
# ↓↓↓ ЗАПОЛНИ ПОД СЕБЯ ↓↓↓
profile_data = {
    "sex": "female", "age": 45, "weight": 80,
    "height": 165, "activity": "light", "goal": "lose",
}
condition  = "diabetes_t2"   # healthy / diabetes_t2 / obesity / ckd / cvd
formula    = "who"
life_stage = "default"

profile = UserProfile(**profile_data)
target = calculate(profile, formula=formula, condition=condition, life_stage=life_stage)
print(f'{target.condition_label} · {target.goal_label}')
print(f'Цель дня: {round(target.target_kcal)} ккал | '
      f'Б {round(target.protein_g)} г | Ж {round(target.fat_g)} г | У {round(target.carbs_g)} г')

## 2. Поиск продукта

Введи название (можно часть, по-русски или по-английски). Таблица показывает варианты.

In [ ]:
query = 'творог'   # например: chicken, рис, кефир, банан, хлеб
found = search(foods, query, limit=15)
found[['name','source','brands','kcal','protein_g','fat_g','carbs_g']]

## 3. Добавить порцию

Укажи **индекс строки** из таблицы поиска выше (0, 1, 2...) и **граммы**. Продукт добавится в дневник, и сразу покажутся предупреждения по болезни для этой порции.

In [ ]:
row_index = 0      # ← номер строки из таблицы поиска
grams     = 150    # ← сколько грамм

item = to_fooditem(found.iloc[row_index])
log.add(item, grams)
print(f'Добавлено: {item.name}, {grams} г')

# Проверки порции против болезни
res = check_portion(item, grams, condition, target_kcal=target.target_kcal)
if res:
    print('Проверки порции:')
    for r in res:
        print(' ', r)
else:
    print('Порция в пределах безопасных лимитов — без предупреждений.')

## 4. Состояние дня

Сводка: сколько съедено vs цель, и предупреждения по накопленным за день микроэлементам.

In [ ]:
print('Съедено за день:')
print(log.to_df().to_string(index=False))
print()
print('Выполнение цели:')
print(log.compare(target).to_string(index=False))
print()
day = log.totals
day_checks = check_day(day, condition, target_kcal=target.target_kcal)
if day_checks:
    print('⚠️ Предупреждения по болезни (превышение лимитов):')
    for r in day_checks:
        print(' ', r)
else:
    print('✅ Лимиты микроэлементов по болезни не превышены.')

## 5. Подбор под норму

Что съесть, чтобы добрать нутриент. Учитывает лимиты текущей болезни.

In [ ]:
nutrient = 'protein_g'   # protein_g / fat_g / carbs_g / kcal
day = log.totals
goals = {'kcal': target.target_kcal, 'protein_g': target.protein_g,
         'fat_g': target.fat_g, 'carbs_g': target.carbs_g}
need = max(goals[nutrient] - day.get(nutrient, 0), 0)
print(f'До нормы по {nutrient} осталось: {need:.0f}')
if need > 0:
    sug = suggest(nutrient, need, condition, top=5, target_kcal=target.target_kcal)
    print(sug.to_string(index=False) if len(sug) else 'Ничего не найдено.')
else:
    print('Норма уже выполнена — добирать не нужно.')

## 6. Управление дневником

- Отменить последнюю запись: `log.pop()`
- Очистить весь день: `log.clear()`

In [ ]:
# раскомментируй при необходимости:
# log.pop()    # убрать последнюю добавленную порцию
# log.clear()  # начать день заново
print(f'Записей в дневнике: {len(log.to_df())}')

## Цикл использования

1. **Поиск** (ячейка 2): ввёл `творог` → таблица
2. **Добавить** (ячейка 3): `row_index=0, grams=150` → проверка порции
3. **Состояние** (ячейка 4): вижу выполнение и предупреждения
4. Если белка мало — **Подбор** (ячейка 5): что добрать
5. Повторяй с ячейки 2 для следующего продукта

Умные предупреждения появляются, когда порция или весь день превышают лимиты болезни (сахар/натрий/калий/фосфор/насыщ.жиры).